# ContractScanner LLM Evaluation, MLflow Tracing, and ROI Comparison

This notebook adds the final LLM and evaluation layer for the ContractScanner AI Agent final project.

The previous notebook prepared the contract analysis examples by loading CUAD contract chunks, retrieving relevant clauses, checking scope, and assigning a basic risk level. This notebook builds on those saved outputs by:

- Running contract questions through GPT-4o and GPT-4o-mini
- Keeping the out-of-scope examples as graceful rejection cases
- Logging five end-to-end examples with MLflow
- Creating evaluation scores for each example
- Explaining how human review fits into the evaluation process
- Comparing GPT-4o and GPT-4o-mini from a business ROI perspective

The goal is to show not only that the agent works technically, but also that it can be evaluated and compared in a way that makes sense for an enterprise legal/compliance use case.

In [0]:
# Install the packages needed for this notebook.
# openai is used for GPT-4o and GPT-4o-mini calls.
# mlflow is used for trace/evaluation logging.
# pandas is used to load and organize the saved agent outputs.

%pip install -q openai mlflow pandas

In [0]:
%pip install -U mlflow
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# Import the Python libraries used throughout this notebook.

import os
import pandas as pd
import time
import mlflow
from openai import OpenAI

In [0]:
# Check the current notebook working directory.
# This helps confirm that Databricks is looking in the correct repo folder.

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

In [0]:
# These are the saved outputs from the previous agent/tool prep notebook.
# The first file contains the 3 contract questions with retrieved contract context.
# The second file contains the 2 graceful rejection examples.

prep_path = "agent_outputs/agent_context_prep_results.csv"
reject_path = "agent_outputs/rejection_examples.csv"

print("Prep exists:", os.path.exists(prep_path))
print("Reject exists:", os.path.exists(reject_path))

In [0]:
# Load the saved examples into pandas DataFrames so we can inspect and reuse them.

prep_df = pd.read_csv(prep_path)
reject_df = pd.read_csv(reject_path)

display(prep_df)
display(reject_df)

In [0]:

def contract_agent(question, retrieved_clause=None, risk_level=None, model_name="gpt-4o-mini"):
	"""
	Full agent workflow:
	1. Checks whether the question is in scope.
	2. Uses retrieved contract context from the tools notebook.
	3. Uses risk level from the risk assessment tool.
	4. Calls the selected LLM to generate a final answer.
	"""
	
	if question in reject_df["question"].values: 
		rejection_answer = reject_df.loc[
			reject_df["question"] == question, "answer"
		].iloc[0]

		return {
			"status": "rejected",
			"question": question,
			"model": "scope_rejection_tool", 
			"answer": rejection_answer,
			"used_tools":["scope_check_tool"]
		}

	try:
		answer, latency = call_contract_llm(
			model_name=model_name,
			question=question,
			retrieved_clause=retrieved_clause,
			risk_level=risk_level
		)

		return {
			"status": "answered",
			"question": question,
			"model": model_name,
			"answer": answer,
			"latency_seconds": latency,
			"used_tools": [
				"contract_retrieval_tool",
				"risk_assessment_tool",
				"llm_response_generator"
			]
		}

	except Exception as e:
		return {
			"status": "error",
			"question": question,
			"model": model_name,
			"answer": f"The agent could not complete the request because of an API or processing error: {str(e)}", 					
            "used_tools": [
				"contract_retrieval_tool",
				"risk_assessment_tool",
				"llm_response_generator"
			]
		}

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("aai510")

In [0]:
# Test OpenAI API access using the secret stored in Databricks.
# This confirms that GPT-4o-mini can be called without putting the API key in the notebook.

from openai import OpenAI
import time

openai_api_key = dbutils.secrets.get(
    scope="aai510",
    key="openai_api_key"
)

client = OpenAI(api_key=openai_api_key)

start_time = time.time()

test_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a contract analysis assistant. Keep answers concise."
        },
        {
            "role": "user",
            "content": "In one sentence, explain what an indemnification clause is."
        }
    ],
    temperature=0.2,
    max_tokens=150
)

latency = round(time.time() - start_time, 3)

print("Model worked.")
print("Latency:", latency, "seconds")
print("Answer:")
print(test_response.choices[0].message.content)

In [0]:
# Test GPT-4o access.
# This confirms that both required models are available before we build the full evaluation.

start_time = time.time()

test_response_4o = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a contract analysis assistant. Keep answers concise."
        },
        {
            "role": "user",
            "content": "In one sentence, explain what a termination clause is."
        }
    ],
    temperature=0.2,
    max_tokens=150
)

latency_4o = round(time.time() - start_time, 3)

print("GPT-4o worked.")
print("Latency:", latency_4o, "seconds")
print("Answer:")
print(test_response_4o.choices[0].message.content)

## Run GPT-4o and GPT-4o-mini on Contract Questions

This section sends the three prepared contract-analysis questions to both GPT-4o and GPT-4o-mini.

The previous notebook already retrieved the most relevant CUAD contract clause and assigned a basic risk level. Here, the LLMs use that retrieved clause as context and generate the final contract-analysis response.

In [0]:
# This function takes one contract question and its retrieved CUAD clause,
# then asks the selected LLM to answer using only that context.

def call_contract_llm(model_name, question, retrieved_clause, risk_level):
    system_prompt = """
You are ContractScanner, an AI contract analysis assistant for legal and compliance teams.

Your job is to analyze contract clauses using the provided retrieved contract context.
You must:
- Answer only based on the retrieved context.
- Identify possible risks clearly.
- Avoid giving final legal advice.
- Recommend human legal/compliance review when needed.
- Keep the answer professional and concise.
"""

    user_prompt = f"""
User question:
{question}

Retrieved contract clause:
{retrieved_clause}

Initial risk level from tool:
{risk_level}

Write a clear contract analysis response.
"""

    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=500
    )

    latency = round(time.time() - start_time, 3)
    answer = response.choices[0].message.content

    return answer, latency

In [0]:

models_to_compare = ["gpt-4o", "gpt-4o-mini"]

llm_results = []

for _, row in prep_df.iterrows():

	question = row["question"]
	retrieved_clause = row["top_retrieved_clause"] 
	risk_level = row["risk_level"]

	for model_name in models_to_compare:

		print(f"Running {model_name} for question: {question}")
		
		result = contract_agent(
			question=question,
			retrieved_clause=retrieved_clause,
			risk_level=risk_level,
			model_name=model_name
		)

		llm_results.append({
			"question": question,
			"model": result["model"],
			"risk_level": risk_level,
			"retrieved_clause": retrieved_clause,
			"answer": result["answer"],
			"latency_seconds": result.get("latency_seconds", None),
			"status": result["status"],
			"used_tools": ", ".join(result["used_tools"])
		})

llm_results_df = pd.DataFrame(llm_results)

display(llm_results_df)

## Evaluation Scoring

This section creates simple evaluation scores for the GPT-4o and GPT-4o-mini responses.

The scoring focuses on:
- Relevance to the user question
- Grounding in the retrieved contract clause
- Risk identification
- Clarity
- Human review recommendation

The scores are not meant to replace a legal expert. They are used as a structured way to compare model outputs before human review.

In [0]:
# Simple evaluation function for contract-analysis responses.

def evaluate_contract_response(question, retrieved_clause, risk_level, answer):
    answer_lower = answer.lower()
    question_lower = question.lower()
    clause_lower = retrieved_clause.lower()

    relevance_score = 5 if any(word in answer_lower for word in question_lower.split()[:4]) else 4

    grounding_score = 5 if any(
        keyword in answer_lower
        for keyword in clause_lower.split()[:12]
    ) else 4

    risk_score = 5 if "risk" in answer_lower or risk_level.lower() in answer_lower else 4

    clarity_score = 5 if len(answer.split()) >= 40 and len(answer.split()) <= 180 else 4

    human_review_score = 5 if "legal" in answer_lower or "review" in answer_lower or "counsel" in answer_lower else 4

    average_score = round(
        (
            relevance_score
            + grounding_score
            + risk_score
            + clarity_score
            + human_review_score
        ) / 5,
        2
    )

    return {
        "relevance_score": relevance_score,
        "grounding_score": grounding_score,
        "risk_score": risk_score,
        "clarity_score": clarity_score,
        "human_review_score": human_review_score,
        "average_score": average_score
    }

In [0]:
# Apply the evaluation function to each LLM response.

evaluation_rows = []

for _, row in llm_results_df.iterrows():
    scores = evaluate_contract_response(
        question=row["question"],
        retrieved_clause=row["retrieved_clause"],
        risk_level=row["risk_level"],
        answer=row["answer"]
    )

    evaluation_rows.append({
        "question": row["question"],
        "model": row["model"],
        "risk_level": row["risk_level"],
        "latency_seconds": row["latency_seconds"],
        "answer": row["answer"],
        **scores
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

## ROI Comparison

This section compares GPT-4o and GPT-4o-mini from a business perspective.

The goal is not only to see which model gives better answers, but also which model makes more sense for a real enterprise deployment. ContractScanner would save money by reducing analyst review time, but the LLM cost still matters when the system is used at scale.

In [0]:
# ROI comparison for GPT-4o vs GPT-4o-mini.
# These prices are written as variables so they can be updated if OpenAI pricing changes.

# Current standard OpenAI API prices per 1M tokens.
# Source: OpenAI pricing page.
MODEL_PRICING = {
    "gpt-4o": {
        "input_cost_per_1m": 2.50,
        "output_cost_per_1m": 10.00
    },
    "gpt-4o-mini": {
        "input_cost_per_1m": 0.15,
        "output_cost_per_1m": 0.60
    }
}

# Business assumptions from the original project proposal.
contracts_per_year = 500
hours_saved_per_contract = 1
analyst_hourly_rate = 75

annual_labor_savings = contracts_per_year * hours_saved_per_contract * analyst_hourly_rate

roi_rows = []

for model_name in ["gpt-4o", "gpt-4o-mini"]:
    model_rows = llm_results_df[llm_results_df["model"] == model_name]

    # Because this notebook used chat.completions, token usage was not saved in llm_results_df.
    # For the ROI estimate, we use a conservative average token estimate per contract question.
    estimated_input_tokens_per_contract = 1000
    estimated_output_tokens_per_contract = 350

    input_cost = (
        estimated_input_tokens_per_contract
        / 1_000_000
        * MODEL_PRICING[model_name]["input_cost_per_1m"]
    )

    output_cost = (
        estimated_output_tokens_per_contract
        / 1_000_000
        * MODEL_PRICING[model_name]["output_cost_per_1m"]
    )

    estimated_cost_per_contract = input_cost + output_cost
    estimated_annual_llm_cost = estimated_cost_per_contract * contracts_per_year
    estimated_net_savings = annual_labor_savings - estimated_annual_llm_cost

    avg_latency = round(model_rows["latency_seconds"].mean(), 3)
    avg_score = round(evaluation_df[evaluation_df["model"] == model_name]["average_score"].mean(), 2)

    roi_rows.append({
        "model": model_name,
        "average_eval_score": avg_score,
        "average_latency_seconds": avg_latency,
        "estimated_cost_per_contract_usd": round(estimated_cost_per_contract, 6),
        "estimated_annual_llm_cost_usd": round(estimated_annual_llm_cost, 2),
        "annual_labor_savings_usd": annual_labor_savings,
        "estimated_net_savings_usd": round(estimated_net_savings, 2)
    })

roi_df = pd.DataFrame(roi_rows)

display(roi_df)

In [0]:
# Run out of scope examples through the agent function.
# These examples are saved separately because they are rejection behavior,
# not grounded contract-answer evaluations.

rejection_agent_rows = []

for _, row in reject_df.iterrows():
	result = contract_agent(question=row["question"])

	rejection_agent_rows.append({
		"question": result["question"],
		"status": result["status"],
		"model": result["model"],
		"answer": result["answer"],
		"used_tools": ", ".join(result["used_tools"])
	})

rejection_agent_df = pd.DataFrame(rejection_agent_rows)

display(rejection_agent_df)

In [0]:
# Save the final evaluation table so it is available in the repo.

rejection_output_path = "agent_outputs/final_rejection_examples.csv"
rejection_agent_df.to_csv(rejection_output_path, index=False)

print("Saved rejection examples to:", rejection_output_path)

## Deployment Recommendation

Based on the evaluation and ROI comparison, GPT-4o-mini is the better default model for ContractScanner because it is much cheaper and fast enough for routine contract review questions. It makes the most sense for first-pass review, especially when many contracts need to be processed.

GPT-4o should still be kept as an escalation model. If a question is high-risk, unclear, or receives a low evaluation score, the system can send the same retrieved context to GPT-4o for a stronger second review.

This hybrid approach gives ContractScanner a better balance of cost, speed, and quality. Routine questions stay low-cost, while more complex contract issues can still use the stronger model when needed.

## Final Notes and Deviations from Proposal

The final project followed the original proposal closely. ContractScanner used CUAD-based contract context, retrieved relevant clauses, assigned risk levels, handled out-of-scope rejection examples, compared GPT-4o and GPT-4o-mini, used MLflow tracing, created evaluation scores, explained human review, and included an ROI comparison.

One small change is that this project used prepared CSV outputs from the earlier agent/tool notebook instead of building a full production upload interface. This kept the project focused on the required agent workflow, model comparison, tracing, evaluation, rejection handling, and ROI analysis.

Overall, GPT-4o-mini is recommended as the default model because it is much cheaper and still performed well on the contract questions. GPT-4o should be kept as an escalation model for higher-risk or unclear contract issues.